# RecoveryAI: research walkthrough

**Primary question.** To what extent can patient characteristics available near hospital admission predict length of
stay in a *synthetic* healthcare dataset, and how reliably can modern ML models quantify uncertainty in those predictions?

**Secondary question.** Do advanced nonlinear models provide statistically meaningful improvement over simple
statistical and naive baselines?

This notebook reads the artefacts written by `python -m training.run_pipeline --budget full`. It does not retrain
anything, and it never touches the test set again.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
R = json.loads((ROOT / "validation/results/validation_report.json").read_text())
pd.set_option("display.precision", 3)
print(R["research_questions"]["primary"])

To what extent can patient characteristics available near hospital admission predict length of stay in a synthetic healthcare dataset, and how reliably can modern machine-learning models quantify uncertainty in those predictions?


## 1. Data: the synthetic LOS distribution
Length_of_Stay = Discharge Date − Date of Admission. Group means are nearly identical, which is a first warning sign.

In [2]:
d = R["data"]
print({k: d[k] for k in ["rows_raw", "exact_duplicates_removed", "n_dev", "n_test", "los_mean", "los_median", "los_sd", "los_skew"]})
pd.DataFrame({k: pd.Series(v) for k, v in d["group_means"].items()}).T

{'rows_raw': 55500, 'exact_duplicates_removed': 534, 'n_dev': 43972, 'n_test': 10994, 'los_mean': 15.499290470472657, 'los_median': 15.0, 'los_sd': 8.66147080729306, 'los_skew': 0.0028326694061938627}


,A+,A-,AB+,AB-,Abnormal,Aetna,Arthritis,Aspirin,Asthma,B+,...,Male,Medicare,Normal,O+,O-,Obesity,Paracetamol,Penicillin,UnitedHealthcare,Urgent
Gender,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,15.536,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Blood Type,15.485,15.435,15.613,15.452,NaN,NaN,NaN,NaN,NaN,15.693,...,NaN,NaN,NaN,15.246,15.547,NaN,NaN,NaN,NaN,NaN
Medical Condition,NaN,NaN,NaN,NaN,NaN,NaN,15.504,NaN,15.677,NaN,...,NaN,NaN,NaN,NaN,NaN,15.448,NaN,NaN,NaN,NaN
Admission Type,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.404
Medication,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.516,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,15.508,15.544,NaN,NaN
Test Results,NaN,NaN,NaN,NaN,15.509,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,15.536,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Insurance Provider,NaN,NaN,NaN,NaN,NaN,15.444,NaN,NaN,NaN,NaN,...,NaN,15.625,NaN,NaN,NaN,NaN,NaN,NaN,15.439,NaN


![LOS](../validation/figures/los_distribution.png)

## 2. Nested cross-validation (headline generalisation estimate)

In [3]:
rows = []
for k, v in R["nested_cv"].items():
    s = v["summary"]
    rows.append({"model": k, "MAE mean": s["mae"]["mean"], "MAE sd": s["mae"]["sd"], "MAE median": s["mae"]["median"],
                 "IQR": s["mae"]["iqr"], "RMSE": s["rmse"]["mean"], "R2": s["r2"]["mean"],
                 "repeated-CV MAE": R["repeated_cv"][k]["summary"]["mae"]["mean"]})
pd.DataFrame(rows).set_index("model")

,MAE mean,MAE sd,MAE median,IQR,RMSE,R2,repeated-CV MAE
model,,,,,,,
median_baseline,7.515,0.029,7.508,0.039,8.678,-3.763e-03,7.517
condition_median,7.513,0.031,7.515,0.026,8.676,-3.379e-03,7.513
gamma_glm,7.515,0.030,7.514,0.037,8.665,-7.719e-04,7.514
xgboost,7.513,0.031,7.507,0.039,8.664,-5.506e-04,7.515
xgboost_aft,7.565,0.036,7.565,0.022,8.760,-2.282e-02,7.541
catboost,7.512,0.031,7.506,0.041,8.666,-9.010e-04,7.514
ft_transformer,7.515,0.031,7.512,0.042,8.666,-8.827e-04,7.513


## 3. Model selection (CV only, locked before testing)

In [4]:
sel = R["selection"]; print(sel["reason"])
pd.DataFrame(sel["comparisons_vs_leader"]).T

7.512-day nested-CV MAE makes 'catboost' the point-estimate leader, but 7 of 7 models are statistically indistinguishable from it (corrected resampled t-test, p > 0.05, or within 0.1 days). By parsimony the simplest of these, 'median_baseline', is selected.


,mean_diff,t,p_value,df,naive_p_value,equivalent_to_best
median_baseline,0.002,0.966,0.35,14,0.054,True
condition_median,-0.001,-0.446,0.662,14,0.348,True
gamma_glm,-0.001,-0.345,0.735,14,0.464,True
xgboost,0.0,0.113,0.912,14,0.809,True
xgboost_aft,0.027,3.712,0.002,14,0.0,True
catboost,0.0,0.0,1.0,14,NaN,True
ft_transformer,-0.001,-0.415,0.684,14,0.381,True


## 4. Final test set: bootstrap CIs and paired differences

In [5]:
T = R["test"]
pd.DataFrame({k: {"MAE": b["metrics"]["mae"]["estimate"], "BCa low": b["metrics"]["mae"]["ci_bca"][0], "BCa high": b["metrics"]["mae"]["ci_bca"][1],
                  "RMSE": b["metrics"]["rmse"]["estimate"], "R2": b["metrics"]["r2"]["estimate"], "bias": b["metrics"]["bias"]["estimate"]}
              for k, b in T["bootstrap"].items()}).T

,MAE,BCa low,BCa high,RMSE,R2,bias
median_baseline,7.510,7.426,7.580,8.673,-3.265e-03,-0.495
condition_median,7.508,7.428,7.580,8.671,-2.869e-03,-0.169
gamma_glm,7.508,7.426,7.579,8.659,2.332e-05,0.007
xgboost,7.508,7.423,7.574,8.663,-9.041e-04,0.057
xgboost_aft,7.549,7.465,7.622,8.735,-1.777e-02,-1.154
catboost,7.512,7.433,7.585,8.663,-1.054e-03,0.044
ft_transformer,7.511,7.429,7.581,8.661,-5.152e-04,-0.152


In [6]:
pd.DataFrame({k: {"MAE diff": v["mae"]["difference"], "CI low": v["mae"]["ci_percentile"][0], "CI high": v["mae"]["ci_percentile"][1],
                  "p": v["mae"]["p_value"]} for k, v in T["paired_differences"].items() if k.endswith("median_baseline")}).T

,MAE diff,CI low,CI high,p
condition_median__vs__median_baseline,-1.637e-03,-0.013,0.009,0.764
gamma_glm__vs__median_baseline,-2.224e-03,-0.012,0.008,0.682
xgboost__vs__median_baseline,-1.925e-03,-0.016,0.011,0.796
xgboost_aft__vs__median_baseline,3.952e-02,0.027,0.053,0.000
catboost__vs__median_baseline,2.669e-03,-0.009,0.014,0.646
ft_transformer__vs__median_baseline,9.548e-04,-0.006,0.008,0.750


![comparison](../validation/figures/model_comparison.png)

## 5. Shuffled-target negative control and permutation p-values

In [7]:
pd.DataFrame({k: {kk: p[kk] for kk in ["observed_mae", "null_mean", "null_sd", "improvement_over_shuffled", "p_value", "n_permutations"]}
              for k, p in R["permutation_tests"].items()}).T

,observed_mae,null_mean,null_sd,improvement_over_shuffled,p_value,n_permutations
condition_median,7.572,7.512,0.042,-0.060,0.915,200.0
gamma_glm,7.566,7.511,0.042,-0.055,0.896,200.0
xgboost,7.564,7.509,0.045,-0.055,0.902,40.0
xgboost_aft,7.584,7.523,0.041,-0.061,1.000,20.0
catboost,7.568,7.490,0.034,-0.078,1.000,20.0
ft_transformer,7.571,7.482,0.037,-0.089,1.000,5.0


![shuffled](../validation/figures/shuffled_target.png)

## 6. Statistical baseline: Gamma GLM (AIC / AICc / BIC are valid only here)

In [8]:
sm = R["statistical_model"]; print(sm["information_criteria"]); print(sm["null_comparison"])
pd.DataFrame(sm["coefficients"]).set_index("term")[["coef", "std_err", "ci_low", "ci_high", "exp_coef", "p_value"]]

{'log_likelihood': -136714.23688308266, 'k_parameters': 25, 'n': 37376, 'aic': 273478.4737661653, 'aicc': 273478.50857205555, 'bic': 273691.69336782, 'deviance': 18806.63733117934, 'pearson_chi2': 11670.07135772036, 'dispersion': 0.3124349795919981}
{'lr_statistic': 26.131486699101515, 'df': 23, 'p_value': 0.2946982734919321, 'null_log_likelihood': -136727.3026264322, 'full_log_likelihood_common_dispersion': -136714.23688308266, 'full_log_likelihood_reported': -136714.23688308266, 'aic_null': 273458.6052528644, 'aic_full': 273478.4737661653, 'bic_null': 273475.6628209968, 'bic_full': 273691.69336782, 'interpretation': 'AIC/BIC lower for the null model means the added predictors do not improve the penalised likelihood enough to justify their parameters.'}


,coef,std_err,ci_low,ci_high,exp_coef,p_value
term,,,,,,
Intercept,2.737e+00,0.014,2.709,2.765,15.445,0.000
Gender[Male],-3.072e-04,0.006,-0.012,0.011,1.000,0.958
Blood Type[A-],2.444e-04,0.012,-0.022,0.023,1.000,0.983
Blood Type[AB+],1.518e-02,0.012,-0.007,0.038,1.015,0.188
Blood Type[AB-],3.459e-03,0.012,-0.019,0.026,1.003,0.764
Blood Type[B+],1.789e-02,0.012,-0.005,0.041,1.018,0.121
Blood Type[B-],-9.730e-04,0.012,-0.024,0.022,0.999,0.933
Blood Type[O+],-1.050e-02,0.012,-0.033,0.012,0.990,0.365
Blood Type[O-],1.307e-02,0.012,-0.010,0.036,1.013,0.258


## 7. Calibration, learning curves and conformal coverage

![calib](../validation/figures/calibration_learning_coverage.png)

## 8. Positive control and quality gate

In [9]:
pc = ROOT / "validation/results/positive_control.json"
if pc.exists():
    P = json.loads(pc.read_text()); print(P["conclusion"]); display(pd.DataFrame({k: {"MAE": v["mae"]["mean"], "R2": v["r2"]["mean"]} for k, v in P["cv"].items()}).T)
g = R["quality_gate"]; print(g["headline"])
pd.DataFrame([{"check": c["label"], "failed": c["failed"]} for c in g["checks"]])

The framework recovers a known, modest admission-time effect: advanced models clearly beat the median baseline and the shuffled-target control. The null result on the real LOS target is therefore a property of the synthetic data, not a failure of the validation pipeline.


,MAE,R2
median_baseline,5.392,-2.146e-04
gamma_glm,4.676,2.513e-01
xgboost,4.693,2.456e-01
catboost,4.676,2.515e-01


Length of stay in this synthetic dataset appears to contain limited predictable signal. Predictions are displayed for methodological demonstration and should not be interpreted as clinically meaningful.


,check,failed
0,Advanced models fail to meaningfully beat the ...,True
1,Bootstrap intervals show no meaningful differe...,True
2,Shuffled-target models perform similarly to re...,True
3,Prediction intervals are excessively wide,True
4,Calibration is poor or predictions are non-dis...,True


## Conclusion

Read the quality gate above together with the positive control. When the framework recovers an injected effect but finds
nothing in the real target, the defensible conclusion is that **this synthetic dataset contains no usable admission-time
signal for length of stay**. A good pipeline should report that rather than hide it.